# 환경설정

In [ ]:
# 인증

from google.colab import auth
auth.authenticate_user()
print("인증 완료")

In [ ]:
# 매직 로드

%load_ext google.cloud.bigquery

# stage.stg_school

**목적**: `accounts_school`를 분석용 Stage로 정제
**산출물**: `stage.stg_school` (VIEW)

**핵심 정보 요약**
- 학생 수가 없는(0명)인 학교 387개(추후 EDA에서 판단)
- 고등학교: 평균(141.3명), 최대(578명), 학교 수(2661개)
- 중학교: 평균(91.5명), 최대(500명), 학교 수(3290개)
- 총 5951개의 행과 id

**Raw 품질**: 중복 0 · 결측 0 · status 예상 외 값 0 · PK(id) 유니크

## Raw 품질 확인

### 데이터 샘플 확인

In [ ]:
%%bigquery --project your-gcp-project
SELECT *
FROM `your-gcp-project.raw.accounts_school`
LIMIT 1;

### 1. 데이터 타입 확인

In [ ]:
%%bigquery --project your-gcp-project
SELECT
    column_name,
    data_type
FROM
    `your-gcp-project.raw.INFORMATION_SCHEMA.COLUMNS`
WHERE
    table_name = 'accounts_school';

### 2. 중복, 결측치, 학교 유형 확인

In [ ]:
%%bigquery --project your-gcp-project
SELECT
  COUNT(*) AS total_rows,
  COUNT(DISTINCT id) AS distinct_id_count,
  -- 결측치(NULL) 검증
  COUNTIF(id IS NULL) AS null_id,
  COUNTIF(address IS NULL) AS null_address,
  COUNTIF(school_type IS NULL) AS null_school_type,
  COUNTIF(student_count IS NULL) AS null_student_count,
  -- 도메인 값 검증 (M, H 외에 다른 값이 섞여 있는지 확인)
  COUNTIF(school_type NOT IN ('M', 'H')) AS invalid_school_type_count
FROM `your-gcp-project.raw.accounts_school`;

### 3. school_type 분포 & student_count 확인

In [ ]:
%%bigquery --project your-gcp-project
SELECT
    school_type,
    COUNT(*) as school_count,
    MIN(student_count) as min_students,
    MAX(student_count) as max_students,
    ROUND(AVG(student_count), 1) as avg_students
FROM `your-gcp-project.raw.accounts_school`
GROUP BY school_type;

### 4. 주소 결측 & 학생 이상치 확인

In [ ]:
%%bigquery --project your-gcp-project
SELECT
    -- 주소 결측치 또는 빈 값 개수
    COUNTIF(address IS NULL OR address = '') AS empty_address_count,
    -- 학생 수가 0명인 학교 개수
    COUNTIF(student_count = 0) AS zero_student_school_count,
    -- 학생 수가 음수인 비정상 데이터 개수
    COUNTIF(student_count < 0) AS negative_student_count
FROM `your-gcp-project.raw.accounts_school`;

### 5. 학생 수가 없는 학교

In [ ]:
%%bigquery --project your-gcp-project
WITH zero_schools AS (
    SELECT *
    FROM `your-gcp-project.raw.accounts_school`
    WHERE student_count = 0
)
-- 1. 학생 수가 0명인 학교의 총 개수 출력
SELECT
    (SELECT COUNT(*) FROM zero_schools) AS total_zero_student_schools,
    id, address, student_count, school_type
FROM zero_schools
LIMIT 10;

## 최종 Stage

### view (관례)

In [ ]:
%%bigquery --project your-gcp-project
CREATE OR REPLACE VIEW `your-gcp-project.stage.stg_school` AS
WITH src AS (
  SELECT
    id,
    address,
    school_type,student_count
  FROM `your-gcp-project.raw.accounts_school`
)
SELECT
  id AS school_id,
  address,
  school_type,
  student_count
FROM src;


### 생성 후 검증

In [ ]:
%%bigquery --project your-gcp-project
SELECT
  COUNT(*) AS total_rows,
  COUNT(DISTINCT school_id) AS distinct_school_id,
  COUNTIF(student_count = 0) AS zero_student_school_rows
FROM `your-gcp-project.raw.stg_votes__school`;